In [ ]:
import os

# Set working directory and GPU before importing JAX
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import vmap

from jaxpi.models import (
    create_lr_schedule,
    create_optimizer,
    create_arch,
    create_train_state,
)
from jaxpi.checkpointing import (
    create_checkpoint_manager,
    save_checkpoint,
    restore_checkpoint,
)

import models
from utils import get_dataset

jax.config.update("jax_default_matmul_precision", "highest")

In [ ]:
# Load config
from configs import baseline, fixed_pseudo_time, pseudo_time
config = baseline.get_config()

In [ ]:
# Get dataset
u_ref, v_ref, t_ref, x_star, y_star, eps, k = get_dataset([0, 0.8])

# convert to shape (num_time_steps, num_x * num_y)
mesh = jnp.stack(jnp.meshgrid(x_star, y_star, indexing="ij"), -1).reshape(-1, 2)
u_ref = u_ref.reshape(len(t_ref), -1)
v_ref = v_ref.reshape(len(t_ref), -1)

# Initial condition of the first time window
u0 = u_ref[0, :]
v0 = v_ref[0, :]

# Get the time domain for each time window
num_time_steps = len(t_ref) // config.training.num_time_windows
t_star = t_ref[:num_time_steps]

# Define the time and space domain
dt = t_star[1] - t_star[0]
t0 = t_star[0]
t1 = t_star[-1] + 1.1 * dt


In [ ]:
# Initialize the model
lr = create_lr_schedule(config.optim)
tx = create_optimizer(config.optim, lr)
arch = create_arch(config.arch)
state = create_train_state(config, tx, arch)

# Initialize model
model = models.GinzburgLandau(config, lr, tx, arch, state, t_max=t1, eps=eps, k=k)

In [ ]:
# Create checkpoint manager for the first time window
window_idx = 1
ckpt_path = os.path.join(os.getcwd(), config.wandb.name, "ckpt")
ckpt_mngr = create_checkpoint_manager(config.saving, ckpt_path, suffix="time_window_{}".format(window_idx))

# restore checkpoint
state = restore_checkpoint(ckpt_mngr, state)
print("Restore checkpoint from step: ", int(state.step))

In [ ]:
# Evaluate on test set
u_preds = []
v_preds = []
for t in t_star:
    print("Evaluating at time: ", t)
    u_pred, v_pred = vmap( model.neural_net, in_axes=(None, None, 0, 0) )(state.params, t, mesh[:, 0], mesh[:, 1])
    u_preds.append(u_pred)
    v_preds.append(v_pred)

u_preds = jnp.stack(u_preds, axis=0)
v_preds = jnp.stack(v_preds, axis=0)

# Compute relative L2 error
u_error = jnp.linalg.norm(u_preds - u_ref[:num_time_steps]) / jnp.linalg.norm(u_ref[:num_time_steps])
v_error = jnp.linalg.norm(v_preds - v_ref[:num_time_steps]) / jnp.linalg.norm(v_ref[:num_time_steps])
print("Relative L2 error for u: ", u_error)
print("Relative L2 error for v: ", v_error)

In [ ]:
# Plot results at the final time step
u_star = u_ref[num_time_steps - 1].reshape(200, 200)
v_star = v_ref[num_time_steps - 1].reshape(200, 200)

u_pred = u_preds[-1, :].reshape(200, 200)
v_pred = v_preds[-1, :].reshape(200, 200)

x_star = mesh[:, 0].reshape(200, 200)
y_star = mesh[:, 1].reshape(200, 200)

plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
plt.title("Reference u")
plt.pcolor(x_star, y_star, u_star, shading="auto", cmap="jet")
plt.colorbar()
plt.subplot(1, 3, 2)
plt.title("Predicted u")
plt.pcolor(x_star, y_star, u_pred, shading="auto", cmap="jet")
plt.colorbar()
plt.subplot(1, 3, 3)
plt.title("Absolute error")
plt.pcolor(x_star, y_star, jnp.abs(u_star - u_pred), shading="auto", cmap="jet")
plt.colorbar()
plt.show()